In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# הגדרת הפחתת שגיאות

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** גרסת הבטא של מודל הרצה חדש זמינה כעת. מודל ההרצה המכוון מספק גמישות רבה יותר בהתאמה אישית של תהליך הפחתת השגיאות. ראו את המדריך [מודל הרצה מכוון](/guides/directed-execution-model) למידע נוסף.


<details>
<summary><b>גרסאות חבילות</b></summary>

הקוד בעמוד זה פותח עם הדרישות הבאות.
אנו ממליצים להשתמש בגרסאות אלו או חדשות יותר.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
טכניקות הפחתת שגיאות מאפשרות למשתמשים להפחית שגיאות במעגלים על ידי
מידול רעש המכשיר בזמן ההרצה. בדרך כלל זה גורר
עלות עיבוד קוונטי מוקדמת הקשורה לאימון המודל
ועלות עיבוד קלאסי מאוחרת להפחתת שגיאות בתוצאות הגולמיות
באמצעות המודל שנוצר.

ה-Estimator primitive תומך בכמה טכניקות הפחתת שגיאות, כולל [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation), [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne), [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec) ו-[PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). ראו [טכניקות הפחתה ודיכוי שגיאות](error-mitigation-and-suppression-techniques) להסבר על כל אחת. כשמשתמשים ב-primitives, אפשר להפעיל או לכבות שיטות בודדות. ראו את הסעיף [הגדרות שגיאה מותאמות אישית](#advanced-error) לפרטים.

> **Note:** Sampler לא תומך בהפחתת שגיאות, אבל אפשר להשתמש בחבילה [mthree](https://qiskit.github.io/qiskit-addon-mthree/) (הפחתת שגיאות מדידה ללא מטריצה) כדי לבצע הפחתת שגיאות מקומית.

Estimator תומך גם ב-`resilience_level`. רמת העמידות מציינת כמה עמידות לבנות מול
שגיאות. רמות גבוהות יותר מייצרות תוצאות מדויקות יותר, על חשבון
זמני עיבוד ארוכים יותר. רמות עמידות אפשר להשתמש בהן להגדרת
פשרת העלות/דיוק בעת יישום הפחתת שגיאות על שאילתת ה-primitive
שלך. הפחתת שגיאות מפחיתה שגיאות (הטיה) בתוצאות על ידי עיבוד
הפלטים מאוסף, או הרכבה, של מעגלים קשורים. מידת
הפחתת השגיאות תלויה בשיטה המיושמת. רמת העמידות
מופשטת מהבחירה המפורטת של שיטת הפחתת השגיאות כדי לאפשר
למשתמשים לשקול את פשרת העלות/דיוק המתאימה
ליישום שלהם.

בהתחשב בכך, כל רמה מתאימה לשיטה אחת או יותר עם
עלות דגימה קוונטית הולכת וגדלה, כדי לאפשר לך להתנסות
עם פשרות שונות בין זמן לדיוק. הטבלה הבאה מראה לך
אילו רמות ושיטות מתאימות זמינות לכל אחד מה-
primitives.

> **Info:** הפחתת שגיאות היא ספציפית למשימה, כך שהטכניקות שאפשר
> ליישם משתנות בהתאם לשאלה אם אתה דוגם התפלגות או מייצר
> ערכי ציפייה.

<span id="resilience-table"></span>

Estimator תומך ברמות העמידות הבאות. Sampler לא תומך ברמות עמידות.

| רמת עמידות | הגדרה                                                                                                    | טכניקה                                                                    |
|-------------|----------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0           | ללא הפחתה                                                                                                | ללא                                                                       |
| 1 [ברירת מחדל] | עלויות הפחתה מינימליות: הפחתת שגיאות הקשורות לשגיאות קריאה          | Twirled Readout Error eXtinction (TREX) ו-measurement twirling            |
| 2           | עלויות הפחתה בינוניות. בדרך כלל מפחית הטיה ב-estimators, אך לא מובטח להיות ללא הטיה. | רמה 1 + Zero Noise Extrapolation (ZNE) ו-gate twirling

> **Info:** רמות העמידות נמצאות כרגע בגרסת בטא, כך שעלות הדגימה
> ואיכות הפתרון ישתנו ממעגל למעגל. תכונות חדשות,
> אפשרויות מתקדמות וכלי ניהול ישוחררו על בסיס מתגלגל.
> שיטות הפחתת שגיאות ספציפיות אינן מובטחות להיות
> מיושמות בכל רמת עמידות.

## הגדרת Estimator עם רמות עמידות
אפשר להשתמש ברמות עמידות כדי לציין טכניקות הפחתת שגיאות, או להגדיר טכניקות מותאמות אישית בנפרד כמתואר ב[הגדרות שגיאה מותאמות אישית.](#advanced-error)

<details>
<summary>רמת עמידות 0</summary>

לא מוחל הפחתת שגיאות על תוכנית המשתמש.

</details>

<details>
<summary>רמת עמידות 1</summary>

רמה 1 מיישמת **הפחתת שגיאות קריאה** ו-**measurement twirling** על ידי שימוש בטכניקה ללא מודל הידועה
כ-Twirled Readout Error eXtinction (TREX). היא מפחיתה שגיאת מדידה
על ידי דיאגונליזציה של ערוץ הרעש הקשור למדידה על ידי
הפיכה אקראית של qubits דרך שערי X מיד לפני המדידה. מונח
שמשמש לכיול מהערוץ הדיאגונלי נלמד על ידי
ביצועי מדידות של מעגלים אקראיים שהאותחלו במצב אפס. זה מאפשר
לשירות להסיר הטיה מערכי ציפייה הנגרמים על ידי
רעש קריאה. גישה זו מתוארת בהרחבה ב-[Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

<details>
<summary>רמת עמידות 2</summary>

רמה 2 מיישמת את **טכניקות הפחתת השגיאות הכלולות ברמה 1** וגם מיישמת **gate twirling** ומשתמשת ב**שיטת Zero Noise Extrapolation (ZNE)**.  ZNE מחשבת
ערך ציפייה של האובייקטיבי עבור גורמי רעש שונים
(שלב הגברה) ואז משתמשת בערכי הציפייה שנמדדו כדי
להסיק את ערך הציפייה האידיאלי בגבול האפס-רעש (שלב
חילוץ). גישה זו נוטה להפחית שגיאות בערכי ציפייה, אך
לא מובטחת להניב תוצאה לא-מוטה.

![תמונה זו מציגה גרף. ציר ה-x מסומן כגורם הגברת רעש. ציר ה-y מסומן כערך ציפייה. קו בעל שיפוע עולה מסומן כערך מוקטן. נקודות קרוב לקו הן ערכים מוגברי-רעש. יש קו אופקי מעט מעל ציר ה-X המסומן כערך מדויק.](../docs/images/guides/configure-error-mitigation/resilience-2.svg "איור של שיטת ZNE")

עלות שיטה זו מתרחבת עם מספר גורמי הרעש. הגדרות
ברירת המחדל דוגמות את ערך הציפייה בשלושה גורמי רעש,
מה שמוביל לעלות של כ-3x בעת שימוש ברמת עמידות זו.

ברמה 2, שיטת TREX הופכת אקראית qubits דרך שערי X מיד לפני המדידה,
ומפכת את הביט המדוד המתאים אם הוחל שער X. גישה זו מתוארת בהרחבה ב-[Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

### דוגמה
ממשק ה-`EstimatorV2` מאפשר למשתמשים לעבוד בצורה חלקה עם מגוון
שיטות הפחתת שגיאות להפחתת שגיאות בערכי ציפייה של
אובייקטיביים. הקוד הבא משתמש ב-Zero Noise Extrapolation ובהפחתת שגיאות קריאה על ידי הגדרה פשוטה של `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## הגדרות שגיאה מותאמות אישית
אפשר להפעיל ולכבות שיטות הפחתה ודיכוי שגיאות בודדות, כולל dynamical decoupling, gate ו-measurement twirling, הפחתת שגיאות מדידה, PEC ו-ZNE. ראו [טכניקות הפחתה ודיכוי שגיאות](error-mitigation-and-suppression-techniques) להסבר על כל אחת.

> **Note:** - לא כל האפשרויות זמינות לשני ה-primitives. ראו את [טבלת האפשרויות הזמינות](runtime-options-overview#options-table) לרשימת האפשרויות הזמינות.
> - לא כל השיטות עובדות יחד על כל סוגי המעגלים. ראו את [טבלת תאימות התכונות](runtime-options-overview#options-compatibility-table) לפרטים.

<Tabs>
  <TabItem value="Estimator" label="Estimator">
    ```python
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}")
print(f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}")
```
  </TabItem>

  <TabItem value="Sampler" label="Sampler">